# ByteEmbed — FINAL retrieval study (BGE-M3 teacher, verified language set)

The finalized retrieval/QA experiment. The byte-vs-subword comparison is entirely within this run —
both students share the teacher, targets, data, recipe, and evaluation; only the tokenizer differs.

**Design (locked — see `RETRIEVAL_EXPERIMENT.md` for the full protocol):**
- **Languages (9):** te, bn, sw, yo, am, ha (lower-resource; bn = Joshi class 3, the one stated
  relaxation — no class 0–2 language with a usable deep benchmark remains) + en, zh, ar (anchors).
  rw/ta/mr/so dropped (no deep-enough benchmark / untrainable wiki).
- **Students:** byt5 vs mt5 × {small, base, large} — 6 models
- **Teacher:** **BGE-M3** (`BAAI/bge-m3`, retrieval-trained, 1024-d), targets cached once
- **Objective (retrieval-only):** pure InfoNCE (τ=0.05, queue 8192) — no alignment term, no
  relational term. AdamW lr 2e-4, **batch 64, 50k steps for every model (iso-step)**, `attn` pooling
- **Eval — ONE deep benchmark per language:** MIRACL dev (te/bn/sw/yo/en/zh/ar) · Amharic-PR (am) ·
  CIRAL Test A (ha, cross-lingual, flagged) — plus Belebele + FLORES (all 9, shallow).
  Mr.TyDi/IndicQA/2AIRTC stay wired, off-default.
- **Baselines:** **BGE-M3 itself (the teacher ceiling)**, mE5-base, LaBSE
- **Optional arms:** `patience` early stop (default off); **boundary injection** `teacher`/`random`
  (byte-only mechanism probe — see the boundary cell below)
- **Caveat to report:** yo is not in XLM-R/CC-100 (the teacher's backbone) — weakest teacher signal;
  both students inherit it equally, so the comparison stays fair.

Everything is resumable; run top-to-bottom; smoke first. First eval streams the CIRAL-ha corpus once
(the big one-time download) — pools cache to `checkpoints/` and every later model reuses them.

### 1. GPU check — confirm you're on an A100 (Runtime → Change runtime type → A100)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

### 2. Clone repo + install deps
Imports work from the repo root even if the editable install is skipped.

In [ ]:
import os
os.chdir('/content')
REPO = 'https://github.com/Aarushvinod/embedding-research.git'
if not os.path.isdir('/content/embedding-research'):
    !git clone -q $REPO
os.chdir('/content/embedding-research')
!git pull -q
!pip install -q -r requirements-cloud.txt
!pip install -q -e . || echo '(editable install skipped — running from repo root is fine)'
print('setup done | cwd', os.getcwd())

### 3. Teacher check + persist to Drive
BGE-M3 loads via sentence-transformers (already a dep) — no extra install, no fairseq2. Point
`PERSIST` at the **same** `byteembed_lowres` folder as the SONAR runs: the balanced-data cache is
reused; the BGE-M3 teacher targets get their own cache file (`teachertargets_bge-m3_*`), so nothing
collides. Skip the Drive block to run on ephemeral disk.

In [ ]:
from huggingface_hub import hf_hub_download
_ = hf_hub_download('BAAI/bge-m3', 'config.json')   # reachable? (weights download on first teacher load)
print('BGE-M3 reachable — teacher will be BAAI/bge-m3 (retrieval-trained, 1024-d)')

from google.colab import drive
drive.mount('/content/drive')
import os, shutil
PERSIST = '/content/drive/MyDrive/byteembed_lowres'   # SAME folder as the SONAR runs -> shared data cache
for d in ('results', 'checkpoints'):
    os.makedirs(f'{PERSIST}/{d}', exist_ok=True)
    if not os.path.islink(d):
        if os.path.isdir(d): shutil.rmtree(d)
        os.symlink(f'{PERSIST}/{d}', d)
print('persisting results/ and checkpoints/ to', PERSIST)

### 4. Smoke test (~5 min) — validate the pipeline with the NEW teacher
3 langs (am/rw/en), 2 tiny students, tiny eval. Confirms: BGE-M3 loads → targets precompute + cache →
train → full eval battery (incl. QA-retrieval) → save. **Check the log says 'BGE-M3 ... loaded' — not
SONAR, not LaBSE.**

In [ ]:
from byte_embed.run_lowresource import run
_ = run(smoke=True, teacher_name='bge-m3', pooling='attn', out='results/retrieval_bgem3_smoke.json')

### 5. Full retrieval study — 6 students, BGE-M3 targets
**50k steps × batch 64 for every model (iso-step), `attn` pooling for all.** The parallel runner
precomputes the BGE-M3 targets once, then trains several students at once. Resumable — finished
models skip; checkpoints carry a `_bge-m3` suffix (`byte-small_attn_bge-m3.pt`) so they never collide
with — or wrongly resume from — the SONAR-run checkpoints.

In [ ]:
from byte_embed.run_parallel import parallel
parallel(
    out='results/retrieval_bgem3.json',
    teacher_name='bge-m3',            # THE one change vs the SONAR study
    pooling='attn',                   # same pooling for byte AND subword (fair)
    steps=50000,                      # iso-step: 50k for every size (a CAP if patience is on)
    max_concurrent=3,                 # 80/96 GB card: 3-5
    patience=0,                       # loss-plateau early stop: 0 = off (exact iso-step).
    # min_delta=1e-3,                 #   e.g. patience=10 -> stop after 10x500 steps w/o improvement;
)                                     #   realized steps land in results as steps_run — report them.

# --- sequential fallback (one model at a time, live logs in-cell):
# from byte_embed.run_lowresource import run
# _ = run(out='results/retrieval_bgem3.json', teacher_name='bge-m3', pooling='attn', steps=50000)

### 5b. OPTIONAL — boundary-injection arms (byte-only mechanism probe)
Arm **B** (`teacher`): markers inserted where BGE-M3's tokenizer would split — segmentation info,
zero vocab table. Arm **C** (`random`): same marker count at random positions — the placebo that
separates "linguistic placement helps" from "any markers help". Teacher targets stay clean; each
arm evals with its own transform. **Sequencing: run B first (byte-small only); add C only if B beats
the raw arm materially.** Separate out file + `_b-{arm}` checkpoint namespace per arm.

In [ ]:
# OPTIONAL boundary arms — byte-small first; uncomment 'random' only if B beats raw materially.
from byte_embed.run_lowresource import run
_ = run(out='results/retrieval_bgem3_bteacher.json', teacher_name='bge-m3', pooling='attn',
        steps=50000, boundary='teacher', only=['byte-small'])          # arm B
# _ = run(out='results/retrieval_bgem3_brandom.json', teacher_name='bge-m3', pooling='attn',
#         steps=50000, boundary='random', only=['byte-small'])         # arm C (placebo)

### 6. Baselines + summary
The parallel runner trains students only. This cell adds the mE5-base + LaBSE baselines to the same
results file (students all skip — already present) and prints the full table incl. the RAG-retrieval
block (IndicQA · Mr.TyDi · Amharic-PR · 2AIRTC · AfriCLIR).

In [ ]:
from byte_embed.run_lowresource import run
_ = run(out='results/retrieval_bgem3.json', teacher_name='bge-m3', pooling='attn', steps=50000)

### 7. Download results
Already on Drive if you ran the persist cell; otherwise grab the JSON here.

In [ ]:
from google.colab import files
files.download('results/retrieval_bgem3.json')